# Supervised Learning in JAX

This notebook demonstrates a complete supervised learning workflow in JAX using logistic regression on a synthetic binary classification problem.

The goal is to show how JAX can be used for numerically stable, vectorized model training with automatic differentiation.

## Problem Setup

Supervised learning uses labeled examples to learn a mapping from inputs to outputs. In this notebook, each training example has two features and a binary label.

We will generate a synthetic dataset with two well-separated classes so the learning process is easy to follow.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(0)
key, class0_key, class1_key, perm_key = jax.random.split(key, 4)
class0 = jax.random.normal(class0_key, (100, 2)) + jnp.array([-2.0, -1.5])
class1 = jax.random.normal(class1_key, (100, 2)) + jnp.array([2.0, 1.5])
X = jnp.concatenate([class0, class1], axis=0)
y = jnp.concatenate([jnp.zeros((100, 1)), jnp.ones((100, 1))], axis=0)
perm = jax.random.permutation(perm_key, X.shape[0])
X = X[perm]
y = y[perm]

## Logistic Regression Model

Logistic regression models the probability of class membership using a linear score passed through a sigmoid function.

JAX makes it straightforward to define the model, compute gradients, and optimize parameters with gradient descent.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + jnp.exp(-z))

def predict(params, inputs):
    weights, bias = params
    return sigmoid(jnp.dot(inputs, weights) + bias)

def loss_fn(params, inputs, targets):
    predictions = predict(params, inputs)
    predictions = jnp.clip(predictions, 1e-7, 1 - 1e-7)
    return -jnp.mean(targets * jnp.log(predictions) + (1 - targets) * jnp.log(1 - predictions))

def accuracy(params, inputs, targets):
    predicted_labels = (predict(params, inputs) >= 0.5).astype(jnp.float32)
    return jnp.mean(predicted_labels == targets)

## Training Loop

We optimize the parameters by repeatedly computing gradients of the loss and updating the weights.

This loop is intentionally kept explicit so that the optimization process is easy to inspect and modify.

In [ ]:
def train_model(inputs, targets, learning_rate=0.1, epochs=200):
    params = (jnp.zeros((inputs.shape[1], 1)), jnp.zeros((1,)))
    gradients = jax.grad(loss_fn)
    history = []

    for epoch in range(epochs):
        grads = gradients(params, inputs, targets)
        params = tuple(param - learning_rate * grad for param, grad in zip(params, grads))
        if epoch % 20 == 0 or epoch == epochs - 1:
            history.append((epoch, float(loss_fn(params, inputs, targets)), float(accuracy(params, inputs, targets))))

    return params, history

params, history = train_model(X, y)
history

## Evaluation and Visualization

After training, we inspect the final metrics and visualize the learned decision behavior on the synthetic data.

A good supervised model should separate the classes with high accuracy and a low training loss.

In [ ]:
final_loss = float(loss_fn(params, X, y))
final_accuracy = float(accuracy(params, X, y))
print(f'Final loss: {final_loss:.4f}')
print(f'Final accuracy: {final_accuracy:.4f}')

weights, bias = params
x_min, x_max = float(jnp.min(X[:, 0])) - 1.0, float(jnp.max(X[:, 0])) + 1.0
y_min, y_max = float(jnp.min(X[:, 1])) - 1.0, float(jnp.max(X[:, 1])) + 1.0
grid_x, grid_y = jnp.meshgrid(jnp.linspace(x_min, x_max, 200), jnp.linspace(y_min, y_max, 200))
grid_points = jnp.stack([grid_x.ravel(), grid_y.ravel()], axis=1)
grid_probs = predict((weights, bias), grid_points).reshape(grid_x.shape)

plt.figure(figsize=(7, 6))
plt.contourf(grid_x, grid_y, grid_probs, levels=25, cmap='RdBu', alpha=0.65)
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap='bwr', edgecolor='black', s=35)
plt.title('Supervised Learning with JAX: Logistic Regression')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()